# 🚀 MorphoSpeech-LLM: State-of-the-Art Turkish Automatic Speech Recognition (ASR)

**MorphoSpeech-LLM** is a novel morphology-aware Audio-LLM framework designed to surpass current SOTA benchmarks on Turkish ASR datasets (**Google FLEURS** and **Mozilla Common Voice**).

### 🌟 Key Innovations:
1. **Turkish Morpho-Tactic Tokenizer (`MorphoTokenizer`)**: Morfessor + BPE integration for agglutinative root-suffix decomposition.
2. **Auxiliary CTC Loss Guidance**: Dual-objective optimization (CTC + Cross-Attention) to ground acoustic phonemes.
3. **Turkish Text Normalization (`TurkishTextNormalizer`)**: Standardized I/İ casing, number-to-words expansion, and punctuation normalization.
4. **PEFT/LoRA Fine-Tuning on NVIDIA A100 GPU**: Memory-efficient, high-speed training.

## 🛠️ Step 1: Environment Setup & Hardware Check

In [6]:
# Check GPU Availability (Target: NVIDIA A100 GPU)
!nvidia-smi

# Install Required Packages (including torchao compatibility fix)
!pip install -q --upgrade transformers datasets[audio] peft bitsandbytes evaluate jiwer morfessor subword-nmt librosa soundfile accelerate huggingface_hub torchao

# Load Hugging Face Access Token from Colab Secrets (🔑 Gizli Anahtarlar tab)
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = token
        from huggingface_hub import login
        login(token=token)
        print('🔑 HF_TOKEN Colab Secrets\'tan başarıyla yüklendi ve HF Hub\'a giriş yapıldı!')
    else:
        print('ℹ️ HF_TOKEN gizli anahtarı boş. Açık verisetleri (Google FLEURS) kullanılacak.')
except Exception as e:
    if os.environ.get('HF_TOKEN'):
        print('🔑 HF_TOKEN ortam değişkeninden yüklendi!')
    else:
        print('ℹ️ HF_TOKEN olmadan anonim olarak devam ediliyor (Google FLEURS kullanılacak).')

Sun Oct  4 21:35:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             50W /  400W |    4054MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


🔑 HF_TOKEN Colab Secrets'tan başarıyla yüklendi ve HF Hub'a giriş yapıldı!


## 📂 Step 2: Project Setup & Repository Cloning

In [7]:
import os
import sys
import shutil

# Clone repository if running directly in Google Colab
if not os.path.exists('src'):
    !git clone https://github.com/gokhanersoy/turkish-asr.git
    %cd turkish-asr
else:
    !git pull

# Auto-cleanup legacy folders if present to prevent Hugging Face datasets and tokenizers module shadowing
for legacy_dir in ['src/datasets', 'src/tokenizers', 'src/tokenizer']:
    if os.path.exists(legacy_dir):
        shutil.rmtree(legacy_dir, ignore_errors=True)
        print(f'🧹 Eski {legacy_dir} klasörü ad çakışmasını engellemek için temizlendi.')

sys.path.append(os.getcwd())
print("Current Working Directory:", os.getcwd())

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 769 bytes | 769.00 KiB/s, done.
From https://github.com/gokhanersoy/turkish-asr
   58a200f..314b04a  main       -> origin/main
Updating 58a200f..314b04a
Fast-forward
 src/models/morpho_speech_llm.py | 31 ++++++++++++++++++++++---------
 1 file changed, 22 insertions(+), 9 deletions(-)
Current Working Directory: /content/turkish-asr


## 📊 Step 3: Baseline Evaluation (Whisper Large-v3 Zero-Shot)
Evaluate baseline model performance on **Google FLEURS (Turkish)**.

In [8]:
import torch
from src.utils.metrics import compute_wer_cer
from src.data.dataset_loader import TurkishASRDatasetLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "openai/whisper-large-v3"

print(f"Loading Baseline Model: {model_id} on {device}...")
processor = WhisperProcessor.from_pretrained(model_id, language="turkish", task="transcribe")
baseline_model = WhisperForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    low_cpu_mem_usage=True
).to(device)

data_loader = TurkishASRDatasetLoader(dataset_name="google/fleurs", language_code="tr_tr")
test_dataset = data_loader.load_dataset_split(split="test", max_samples=50)

preds, refs = [], []
for sample in tqdm(test_dataset, desc="Evaluating Baseline Zero-Shot"):
    audio = sample["audio"]
    ref_text = sample.get("transcription") or sample.get("raw_transcription") or sample.get("sentence") or sample.get("text") or ""
    input_features = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt").input_features.to(device).half()
    with torch.no_grad():
        predicted_ids = baseline_model.generate(input_features, forced_decoder_ids=processor.get_decoder_prompt_ids(language="turkish", task="transcribe"))
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    preds.append(transcription)
    refs.append(ref_text)

baseline_metrics = compute_wer_cer(preds, refs)
print(f"\nBaseline Zero-Shot Results (Google FLEURS Turkish):")
print(f" - WER: {baseline_metrics['wer']}%")
print(f" - CER: {baseline_metrics['cer']}%")

Loading Baseline Model: openai/whisper-large-v3 on cuda...


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Evaluating Baseline Zero-Shot: 100%|██████████| 50/50 [01:19<00:00,  1.59s/it]


Baseline Zero-Shot Results (Google FLEURS Turkish):
 - WER: 7.47%
 - CER: 1.28%


## 🏋️ Step 4: Fine-Tuning MorphoSpeech-LLM on NVIDIA A100 GPU
Train MorphoSpeech-LLM using PEFT/LoRA and Auxiliary CTC loss on **Google FLEURS Turkish ('google/fleurs', 'tr_tr')**.

In [9]:
!PYTHONPATH=. python src/train.py \
    --model_name_or_path openai/whisper-large-v3 \
    --dataset_name google/fleurs \
    --dataset_config tr_tr \
    --output_dir ./checkpoints/morpho_speech_llm_a100 \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 16 \
    --learning_rate 2e-4 \
    --num_train_epochs 3 \
    --gradient_accumulation_steps 2 \
    --fp16 \
    --use_lora

 MorphoSpeech-LLM Turkish ASR Pipeline Launch
 Base Model: openai/whisper-large-v3
 Dataset: google/fleurs (tr_tr)
 CUDA Available: True
 GPU Device: NVIDIA A100-SXM4-40GB
Loaded 743 test samples for evaluation.
Loaded 2526 train samples.
Loading weights: 100% 1259/1259 [00:00<00:00, 4951.04it/s]
trainable params: 28,835,840 || all params: 1,572,326,400 || trainable%: 1.8340
2026-10-04 21:37:34.570741: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-04 21:37:34.634146: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler fla

## 🏆 Step 5: Benchmark & SOTA Comparison
Compare MorphoSpeech-LLM results against existing literature SOTA values.

In [10]:
import pandas as pd

sota_results = [
    {"Model": "Whisper Large-v3 (Zero-Shot)", "Dataset": "FLEURS (TR)", "WER (%)": 7.47, "Paper/Source": "OpenAI (2023)"},
    {"Model": "Meta MMS-1B", "Dataset": "FLEURS (TR)", "WER (%)": 7.2, "Paper/Source": "Meta AI (2023)"},
    {"Model": "Whisper-Small LoRA", "Dataset": "Common Voice (TR)", "WER (%)": 16.0, "Paper/Source": "DergiPark (2023)"},
    {"Model": "HuBERT-TR", "Dataset": "Broadcast News", "WER (%)": 4.97, "Paper/Source": "HuggingFace (2023)"},
    {"Model": "MorphoSpeech-LLM (Ours)", "Dataset": "Google FLEURS (TR)", "WER (%)": "< 5.5 (Target)", "Paper/Source": "MorphoSpeech-LLM Framework"}
]

df = pd.DataFrame(sota_results)
print(df.to_markdown(index=False))

| Model                        | Dataset            | WER (%)        | Paper/Source               |
|:-----------------------------|:-------------------|:---------------|:---------------------------|
| Whisper Large-v3 (Zero-Shot) | FLEURS (TR)        | 7.47           | OpenAI (2023)              |
| Meta MMS-1B                  | FLEURS (TR)        | 7.2            | Meta AI (2023)             |
| Whisper-Small LoRA           | Common Voice (TR)  | 16.0           | DergiPark (2023)           |
| HuBERT-TR                    | Broadcast News     | 4.97           | HuggingFace (2023)         |
| MorphoSpeech-LLM (Ours)      | Google FLEURS (TR) | < 5.5 (Target) | MorphoSpeech-LLM Framework |
